In [0]:
dbutils.widgets.text("database_name", "mf_insight", "Database (schema) name")
dbutils.widgets.text("dip_threshold_pct", "-10", "Dip threshold (% drawdown from rolling peak)")
dbutils.widgets.text("hike_threshold_pct", "10", "Hike threshold (% rise from rolling trough)")
dbutils.widgets.text("min_duration_days", "5", "Minimum trading days sustained to count as an event")

DATABASE_NAME = dbutils.widgets.get("database_name")
DIP_THRESHOLD = float(dbutils.widgets.get("dip_threshold_pct"))
HIKE_THRESHOLD = float(dbutils.widgets.get("hike_threshold_pct"))
MIN_DURATION_DAYS = int(dbutils.widgets.get("min_duration_days"))

spark.sql(f"USE {DATABASE_NAME}")

print(f"Database       : {DATABASE_NAME}")
print(f"Dip threshold  : {DIP_THRESHOLD}% drawdown from rolling peak")
print(f"Hike threshold : {HIKE_THRESHOLD}% rise from rolling trough")
print(f"Min duration   : {MIN_DURATION_DAYS} trading days")

Database       : mf_insight
Dip threshold  : -10.0% drawdown from rolling peak
Hike threshold : 10.0% rise from rolling trough
Min duration   : 5 trading days


In [0]:
nav_df = spark.table(f"{DATABASE_NAME}.nav_history")
nav_df.createOrReplaceTempView("nav_base")
print(f"Loaded {nav_df.count()} NAV rows across {nav_df.select('scheme_code').distinct().count()} funds")

Loaded 58606 NAV rows across 16 funds


In [0]:
bad_nav_df = nav_df.filter("nav <= 0")
print(f"Rows with NAV <= 0 (invalid): {bad_nav_df.count()}")
display(bad_nav_df.select("scheme_code", "scheme_name", "nav_date", "nav"))

nav_df = nav_df.filter("nav > 0")
nav_df.createOrReplaceTempView("nav_base")
print(f"Clean NAV rows after filtering: {nav_df.count()}")

Rows with NAV <= 0 (invalid): 5


scheme_code,scheme_name,nav_date,nav
112354,Axis Short Duration Fund - Regular Plan - Growth Option,2012-06-03,0.0
114564,Axis Midcap Fund - Regular Plan - Growth,2013-04-07,0.0
112277,Axis Large Cap Fund - Regular Plan - Growth,2013-04-07,0.0
120505,Axis Midcap Fund - Direct Plan - Growth,2013-04-07,0.0
120465,Axis Large Cap Fund - Direct Plan - Growth,2013-04-07,0.0


Clean NAV rows after filtering: 58601


In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW nav_with_extremes AS
SELECT
  scheme_code,
  scheme_name,
  nav_date,
  nav,
  MAX(nav) OVER (PARTITION BY scheme_code ORDER BY nav_date
                 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_peak,
  MIN(nav) OVER (PARTITION BY scheme_code ORDER BY nav_date
                 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_trough
FROM nav_base
""")
print("nav_with_extremes view created")
spark.sql("SELECT * FROM nav_with_extremes ORDER BY scheme_code, nav_date").show(5)

nav_with_extremes view created
+-----------+--------------------+----------+-----+------------+--------------+
|scheme_code|         scheme_name|  nav_date|  nav|running_peak|running_trough|
+-----------+--------------------+----------+-----+------------+--------------+
|     101592|Aditya Birla Sun ...|2006-04-03|58.86|       58.86|         58.86|
|     101592|Aditya Birla Sun ...|2006-04-04|59.56|       59.56|         58.86|
|     101592|Aditya Birla Sun ...|2006-04-05|60.42|       60.42|         58.86|
|     101592|Aditya Birla Sun ...|2006-04-07|59.67|       60.42|         58.86|
|     101592|Aditya Birla Sun ...|2006-04-10|59.74|       60.42|         58.86|
+-----------+--------------------+----------+-----+------------+--------------+
only showing top 5 rows


In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW nav_with_pct AS
SELECT
  *,
  ROUND(try_divide((nav - running_peak), running_peak) * 100, 2) AS drawdown_pct,
  ROUND(try_divide((nav - running_trough), running_trough) * 100, 2) AS rise_pct
FROM nav_with_extremes
""")
print("nav_with_pct view created")
spark.sql("SELECT scheme_code, nav_date, nav, drawdown_pct, rise_pct FROM nav_with_pct ORDER BY drawdown_pct ASC").show(5)

nav_with_pct view created
+-----------+----------+-----+------------+--------+
|scheme_code|  nav_date|  nav|drawdown_pct|rise_pct|
+-----------+----------+-----+------------+--------+
|     101592|2009-03-09|38.07|      -68.28|     0.0|
|     101592|2009-03-12|38.16|       -68.2|    0.24|
|     101592|2009-03-06|38.77|      -67.69|     0.0|
|     101592|2009-03-05|38.91|      -67.58|     0.0|
|     101592|2009-03-13|39.11|      -67.41|    2.73|
+-----------+----------+-----+------------+--------+
only showing top 5 rows


In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW nav_flagged AS
SELECT
  *,
  CASE WHEN drawdown_pct <= {DIP_THRESHOLD} THEN 1 ELSE 0 END AS is_dip,
  CASE WHEN rise_pct >= {HIKE_THRESHOLD} THEN 1 ELSE 0 END AS is_hike
FROM nav_with_pct
""")
flagged_counts = spark.sql("SELECT SUM(is_dip) AS dip_days, SUM(is_hike) AS hike_days FROM nav_flagged").first()
print(f"Flagged dip-days: {flagged_counts['dip_days']}  |  Flagged hike-days: {flagged_counts['hike_days']}")

Flagged dip-days: 9548  |  Flagged hike-days: 55276


In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW dip_groups AS
SELECT
  *,
  SUM(CASE WHEN is_dip = 1
            AND COALESCE(LAG(is_dip) OVER (PARTITION BY scheme_code ORDER BY nav_date), 0) = 0
       THEN 1 ELSE 0 END)
    OVER (PARTITION BY scheme_code ORDER BY nav_date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS dip_group_id
FROM nav_flagged
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW dip_events AS
SELECT
  scheme_code,
  'DIP' AS event_type,
  MIN(nav_date) AS start_date,
  MAX(nav_date) AS end_date,
  COUNT(*) AS duration_days,
  ROUND(MIN(drawdown_pct), 2) AS magnitude_pct
FROM dip_groups
WHERE is_dip = 1
GROUP BY scheme_code, dip_group_id
HAVING COUNT(*) >= {MIN_DURATION_DAYS}
""")

n_dips = spark.table("dip_events").count()
print(f"Dip episodes found (>= {MIN_DURATION_DAYS} trading days, <= {DIP_THRESHOLD}%): {n_dips}")
spark.sql("SELECT * FROM dip_events ORDER BY magnitude_pct ASC").show(10)

Dip episodes found (>= 5 trading days, <= -10.0%): 180
+-----------+----------+----------+----------+-------------+-------------+
|scheme_code|event_type|start_date|  end_date|duration_days|magnitude_pct|
+-----------+----------+----------+----------+-------------+-------------+
|     101592|       DIP|2008-01-18|2010-01-08|          478|       -68.28|
|     103174|       DIP|2008-01-18|2009-10-12|          420|       -56.37|
|     103155|       DIP|2008-01-21|2009-09-14|          401|       -49.81|
|     101592|       DIP|2018-09-04|2020-12-11|          557|       -46.15|
|     119620|       DIP|2018-09-04|2020-11-27|          548|       -45.08|
|     103174|       DIP|2020-03-09|2020-08-06|          102|       -37.73|
|     119528|       DIP|2020-03-09|2020-07-22|           91|       -37.66|
|     101592|       DIP|2006-05-18|2006-09-27|           94|        -35.3|
|     101592|       DIP|2011-01-07|2012-12-13|          477|       -33.07|
|     103155|       DIP|2020-03-06|2020-08-07

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW hike_groups AS
SELECT
  *,
  SUM(CASE WHEN is_hike = 1
            AND COALESCE(LAG(is_hike) OVER (PARTITION BY scheme_code ORDER BY nav_date), 0) = 0
       THEN 1 ELSE 0 END)
    OVER (PARTITION BY scheme_code ORDER BY nav_date
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS hike_group_id
FROM nav_flagged
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW hike_events AS
SELECT
  scheme_code,
  'HIKE' AS event_type,
  MIN(nav_date) AS start_date,
  MAX(nav_date) AS end_date,
  COUNT(*) AS duration_days,
  ROUND(MAX(rise_pct), 2) AS magnitude_pct
FROM hike_groups
WHERE is_hike = 1
GROUP BY scheme_code, hike_group_id
HAVING COUNT(*) >= {MIN_DURATION_DAYS}
""")

n_hikes = spark.table("hike_events").count()
print(f"Hike episodes found (>= {MIN_DURATION_DAYS} trading days, >= {HIKE_THRESHOLD}%): {n_hikes}")
spark.sql("SELECT * FROM hike_events ORDER BY magnitude_pct DESC").show(10)

Hike episodes found (>= 5 trading days, >= 10.0%): 41
+-----------+----------+----------+----------+-------------+-------------+
|scheme_code|event_type|start_date|  end_date|duration_days|magnitude_pct|
+-----------+----------+----------+----------+-------------+-------------+
|     101592|      HIKE|2009-03-26|2026-06-23|         4242|      2112.32|
|     103174|      HIKE|2006-06-21|2026-06-23|         4924|      1614.85|
|     114564|      HIKE|2012-01-31|2026-06-23|         3549|       1272.6|
|     120505|      HIKE|2013-10-04|2026-06-23|         3134|      1199.08|
|     103155|      HIKE|2009-03-13|2026-06-23|         4252|      1118.99|
|     119620|      HIKE|2013-09-25|2026-06-23|         3131|       933.68|
|     112277|      HIKE|2012-06-06|2026-06-23|         3463|       611.56|
|     119528|      HIKE|2013-10-01|2026-06-23|         3127|       588.69|
|     120465|      HIKE|2013-10-03|2026-06-23|         3135|       531.06|
|     120517|      HIKE|2013-10-09|2026-06-23|

In [0]:
fund_events_df = spark.sql("""
SELECT scheme_code, event_type, start_date, end_date, duration_days, magnitude_pct FROM dip_events
UNION ALL
SELECT scheme_code, event_type, start_date, end_date, duration_days, magnitude_pct FROM hike_events
ORDER BY scheme_code, start_date
""")

(fund_events_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{DATABASE_NAME}.fund_events"))

print(f"FUND_EVENTS written: {fund_events_df.count()} rows ({n_dips} dips, {n_hikes} hikes)")
display(fund_events_df)

FUND_EVENTS written: 221 rows (180 dips, 41 hikes)


scheme_code,event_type,start_date,end_date,duration_days,magnitude_pct
101592,DIP,2006-05-18,2006-09-27,94,-35.3
101592,HIKE,2006-06-16,2008-10-23,587,194.62
101592,DIP,2007-02-28,2007-04-12,30,-16.28
101592,DIP,2008-01-18,2010-01-08,478,-68.28
101592,HIKE,2008-10-31,2008-11-18,12,22.29
101592,HIKE,2008-12-12,2009-01-09,19,21.9
101592,HIKE,2009-03-26,2026-06-23,4242,2112.32
101592,DIP,2010-01-20,2010-04-01,48,-18.97
101592,DIP,2010-05-04,2010-06-29,41,-17.47
101592,DIP,2010-07-01,2010-07-07,5,-10.38


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Large date gaps (beyond normal weekend/holiday gaps)
w = Window.partitionBy("scheme_code").orderBy("nav_date")
gap_df = (nav_df
    .withColumn("prev_date", F.lag("nav_date").over(w))
    .withColumn("gap_days", F.datediff("nav_date", "prev_date"))
    .filter(F.col("gap_days") > 10))
print(f"Date gaps > 10 days: {gap_df.count()}")
display(gap_df.select("scheme_code", "scheme_name", "prev_date", "nav_date", "gap_days").orderBy(F.desc("gap_days")))

# 2. Duplicate (scheme_code, nav_date) pairs
dup_df = nav_df.groupBy("scheme_code", "nav_date").count().filter("count > 1")
print(f"Duplicate scheme_code+nav_date pairs: {dup_df.count()}")
display(dup_df)

# 3. Scheme codes in NAV_HISTORY with no matching FUND_MASTER row
fund_master_df = spark.table(f"{DATABASE_NAME}.fund_master")
orphan_codes = (nav_df.select("scheme_code").distinct()
    .join(fund_master_df.select("scheme_code"), "scheme_code", "left_anti"))
print(f"Orphan scheme codes (no AMC match): {orphan_codes.count()}")
display(orphan_codes)

Date gaps > 10 days: 0


scheme_code,scheme_name,prev_date,nav_date,gap_days


Duplicate scheme_code+nav_date pairs: 0


scheme_code,nav_date,count


Orphan scheme codes (no AMC match): 0


scheme_code


In [0]:
print("=== End of Day 2 Person A checkpoint ===")
print(f"FUND_EVENTS table : {fund_events_df.count()} events ({n_dips} dips, {n_hikes} hikes)")
print(f"Date gaps flagged  : {gap_df.count()}")
print(f"Duplicates flagged : {dup_df.count()}")
print(f"Orphan codes       : {orphan_codes.count()}")

=== End of Day 2 Person A checkpoint ===
FUND_EVENTS table : 221 events (180 dips, 41 hikes)
Date gaps flagged  : 0
Duplicates flagged : 0
Orphan codes       : 0


In [0]:
for t in ["fund_master", "nav_history", "fund_events"]:
    n = spark.table(f"mf_insight.{t}").count()
    print(f"{t:15s} -> {n} rows")

fund_master     -> 16 rows
nav_history     -> 58606 rows
fund_events     -> 221 rows
